# Verifikasi implementasi SMOTE dan SMOTE-NC terhadap imbalanced-learn
Menjawab komentar reviewer #28. Notebook ini membandingkan SMOTE dan SMOTE-NC yang diimplementasikan sendiri pada naskah dengan implementasi pustaka **imbalanced-learn** pada data, praproses, dan model yang sama.

**Desain:** validasi silang stratified 5-fold, diulang pada 5 *seed* (42–46), untuk tiga model (LR, RF, HGB) dengan parameter bawaan. Setiap kombinasi dijalankan dengan dua implementasi, dan selisih antarimplementasi dibandingkan dengan variasi antar-*seed* pada implementasi yang sama.

**Cara menjalankan:** letakkan notebook ini satu folder dengan `healthcare-dataset-stroke-data.csv` (atau jalankan dari folder `notebooks/` repositori), lalu jalankan semua sel secara berurutan (Run All). Perkiraan waktu: 10–30 menit. Hasil disimpan di `hasil_verifikasi_imblearn/` (atau `results/revisi1/verifikasi_imblearn/` bila dijalankan dari folder `notebooks/` repositori).

In [1]:
# Jalankan sekali jika imbalanced-learn belum terpasang
# pip install -q imbalanced-learn

## 1. Data, praproses, dan implementasi sendiri (identik dengan notebook naskah)

In [2]:
# Fungsi bersama, identik dengan notebook 01/02 (stroke_leakage_JIEE / stroke_revisi_M1_M3)
import os, json, time, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV, train_test_split, cross_val_predict
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef, recall_score,
                             precision_score, f1_score, brier_score_loss, roc_curve, confusion_matrix)
SEED = 42; N_REPEATS = 2; N_ITER = 12; N_BOOT = 1000
def _cari(*opsi):
    for o in opsi:
        if os.path.exists(o): return os.path.abspath(o)
    raise FileNotFoundError(f"Tidak ditemukan: {opsi}")
# bekerja dari folder notebooks/ repositori maupun dari folder kerja yang berisi dataset
DATA = _cari("healthcare-dataset-stroke-data.csv", "../data/healthcare-dataset-stroke-data.csv", "data/healthcare-dataset-stroke-data.csv")
df = pd.read_csv(DATA).drop(columns="id")
df = df[df.gender != "Other"].reset_index(drop=True)
y = df.pop("stroke").values; X = df.copy()
X["bmi_kosong"] = X.bmi.isna().astype(int)          # hanya dipakai pada varian indikator
NUM = ["age", "avg_glucose_level", "bmi"]; BIN = ["hypertension", "heart_disease"]
CAT = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
CATS = [sorted(X[c].unique()) for c in CAT]
def make_prep(indicator=False):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
        ("bin", "passthrough", BIN + (["bmi_kosong"] if indicator else [])),
        ("cat", OneHotEncoder(categories=CATS, drop="first", handle_unknown="ignore", sparse_output=False), CAT)])
NUM_IDX = list(range(len(NUM))); GROUPS = [[len(NUM)], [len(NUM) + 1]]; _s = len(NUM) + 2
for c in CATS: GROUPS.append(list(range(_s, _s + len(c) - 1))); _s += len(c) - 1
def smote(X, y, k=5, seed=0):
    r = np.random.RandomState(seed); Xm = X[y == 1]; n = int((y == 0).sum() - len(Xm))
    if n <= 0: return X, y
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(Xm))).fit(Xm)
    idx = nn.kneighbors(Xm, return_distance=False)[:, 1:]
    i = r.randint(0, len(Xm), n); j = idx[i, r.randint(0, idx.shape[1], n)]; g = r.rand(n, 1)
    return np.vstack([X, Xm[i] + g * (Xm[j] - Xm[i])]), np.r_[y, np.ones(n, int)]
def rus(X, y, seed=0):
    r = np.random.RandomState(seed); pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    keep = np.r_[pos, r.choice(neg, len(pos), replace=False)]; return X[keep], y[keep]
def ros(X, y, seed=0):
    r = np.random.RandomState(seed); pos = np.where(y == 1)[0]; n = int((y == 0).sum() - len(pos))
    add = r.choice(pos, n, replace=True); return np.vstack([X, X[add]]), np.r_[y, np.ones(n, int)]
def smotenc(X, y, k=5, seed=0):
    r = np.random.RandomState(seed); Xm = X[y == 1]; n = int((y == 0).sum() - len(Xm))
    if n <= 0: return X, y
    med_sd = np.median(Xm[:, NUM_IDX].std(axis=0))
    dn = ((Xm[:, None, NUM_IDX] - Xm[None, :, NUM_IDX]) ** 2).sum(-1)
    dc = sum((np.abs(Xm[:, None, g] - Xm[None, :, g]).sum(-1) > 0).astype(float) for g in GROUPS)
    D = np.sqrt(dn + dc * med_sd ** 2); np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1)[:, :k]
    i = r.randint(0, len(Xm), n); j = nn[i, r.randint(0, k, n)]; gap = r.rand(n, 1)
    S = Xm[i].copy(); S[:, NUM_IDX] = Xm[i][:, NUM_IDX] + gap * (Xm[j][:, NUM_IDX] - Xm[i][:, NUM_IDX])
    for g in GROUPS:
        neigh = Xm[:, g][nn[i]]
        for t in range(n):
            vals, cnt = np.unique(neigh[t], axis=0, return_counts=True); S[t, g] = vals[cnt.argmax()]
    return np.vstack([X, S]), np.r_[y, np.ones(n, int)]
class Resampled(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator=None, sampler="none", random_state=0):
        self.estimator = estimator; self.sampler = sampler; self.random_state = random_state
    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        f = {"smote": smote, "rus": rus, "smotenc": smotenc, "ros": ros}.get(self.sampler)
        if f: X, y = f(X, y, seed=self.random_state)
        self.est_ = clone(self.estimator).fit(X, y); self.classes_ = self.est_.classes_; return self
    def predict_proba(self, X): return self.est_.predict_proba(np.asarray(X, float))
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def _fit(Xd, yy, off=None, it=60):
    b = np.zeros(Xd.shape[1]); off = np.zeros(len(yy)) if off is None else off
    for _ in range(it):
        mu = 1 / (1 + np.exp(-(Xd @ b + off))); W = mu * (1 - mu)
        b = b + np.linalg.solve(Xd.T @ (Xd * W[:, None]) + 1e-9 * np.eye(Xd.shape[1]), Xd.T @ (yy - mu))
    return b
def calib(p, yy):
    lp = logit(p); a = _fit(np.ones((len(yy), 1)), yy, off=lp)[0]; s = _fit(np.c_[np.ones(len(yy)), lp], yy)[1]; return a, s
import imblearn, sklearn
from imblearn.over_sampling import SMOTE as IMB_SMOTE, SMOTENC as IMB_SMOTENC
from sklearn.preprocessing import OrdinalEncoder
OUT = "../results/revisi1/verifikasi_imblearn" if os.path.isdir("../results") else "hasil_verifikasi_imblearn"; os.makedirs(OUT, exist_ok=True)
print("hasil disimpan di", os.path.abspath(OUT))
print("imbalanced-learn", imblearn.__version__, "| scikit-learn", sklearn.__version__, "| data", X.shape, "| stroke", y.sum())

hasil disimpan di results/revisi1/verifikasi_imblearn
imbalanced-learn 0.14.2 | scikit-learn 1.9.1 | data (5109, 11) | stroke 249


## 2. Pembungkus imbalanced-learn
- **SMOTE**: dijalankan pada ruang fitur hasil praproses yang sama dengan implementasi sendiri (numerik terstandardisasi + biner + one-hot).
- **SMOTE-NC**: imbalanced-learn bekerja pada atribut kategorikal asli, sehingga data diubah dulu menjadi numerik terstandardisasi + kode kategori (ordinal), di-*resample* dengan SMOTENC, lalu atribut kategorikal di-*one-hot* sebelum masuk model. Atribut biner (hipertensi, penyakit jantung) diperlakukan sebagai kategorikal, sama seperti implementasi sendiri.

In [3]:
class ImbSMOTE(ClassifierMixin, BaseEstimator):
    """praproses (dipelajari dari data latih) -> SMOTE imbalanced-learn -> model"""
    def __init__(self, estimator=None, random_state=0): self.estimator = estimator; self.random_state = random_state
    def fit(self, X_, y_):
        self.prep_ = make_prep().fit(X_)
        Xr, yr = IMB_SMOTE(k_neighbors=5, random_state=self.random_state).fit_resample(self.prep_.transform(X_), np.asarray(y_))
        self.Xs_, self.ys_ = Xr, yr
        self.est_ = clone(self.estimator).fit(Xr, yr); self.classes_ = self.est_.classes_; return self
    def predict_proba(self, X_): return self.est_.predict_proba(self.prep_.transform(X_))

class ImbSMOTENC(ClassifierMixin, BaseEstimator):
    """numerik terstandardisasi + kode kategori -> SMOTENC imbalanced-learn -> one-hot -> model"""
    def __init__(self, estimator=None, random_state=0): self.estimator = estimator; self.random_state = random_state
    def fit(self, X_, y_):
        self.ord_ = ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
            ("bc", OrdinalEncoder(categories=[[0, 1], [0, 1]] + CATS), BIN + CAT)]).fit(X_)
        Z = self.ord_.transform(X_); cat_idx = list(range(len(NUM), Z.shape[1]))
        Zr, yr = IMB_SMOTENC(categorical_features=cat_idx, k_neighbors=5, random_state=self.random_state).fit_resample(Z, np.asarray(y_))
        self.ohe_ = OneHotEncoder(categories=[list(range(len(c))) for c in CATS], drop="first", sparse_output=False).fit(Zr[:, len(NUM) + len(BIN):])
        Xr = self._enc(Zr); self.Xs_, self.ys_ = Xr, yr
        self.est_ = clone(self.estimator).fit(Xr, yr); self.classes_ = self.est_.classes_; return self
    def _enc(self, Z): return np.hstack([Z[:, :len(NUM) + len(BIN)], self.ohe_.transform(Z[:, len(NUM) + len(BIN):])])
    def predict_proba(self, X_): return self.est_.predict_proba(self._enc(self.ord_.transform(X_)))

def own(estimator, sampler, seed):
    return Pipeline([("prep", make_prep()), ("clf", Resampled(estimator, sampler, seed))])

MODELS = {"LR": LogisticRegression(max_iter=3000),
          "RF": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
          "HGB": HistGradientBoostingClassifier(random_state=SEED)}
VARIAN = {("SMOTE", "implementasi sendiri"): lambda m, s: own(m, "smote", s),
          ("SMOTE", "imbalanced-learn"): lambda m, s: ImbSMOTE(m, s),
          ("SMOTE-NC", "implementasi sendiri"): lambda m, s: own(m, "smotenc", s),
          ("SMOTE-NC", "imbalanced-learn"): lambda m, s: ImbSMOTENC(m, s)}

## 3. Pemeriksaan sampel sintetis (satu kali resampling pada seluruh data latih fold pertama)

In [4]:
# Proporsi nilai pecahan pada kolom one-hot sampel sintetis dan jumlah kelas setelah resampling
tr, te = next(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y))
cek = []
for (alg, impl), f in VARIAN.items():
    m = f(LogisticRegression(max_iter=3000), SEED).fit(X.iloc[tr], y[tr])
    if impl == "implementasi sendiri":
        Xp = m.named_steps["prep"].transform(X.iloc[tr]); Xs, ys = {"SMOTE": smote, "SMOTE-NC": smotenc}[alg](Xp, y[tr], seed=SEED)
    else:
        Xs, ys = m.Xs_, m.ys_
    syn = Xs[len(tr):]; oh = syn[:, len(NUM):]
    cek.append(dict(Algoritma=alg, Implementasi=impl, n_latih=len(ys), n_positif=int(ys.sum()), n_sintetis=len(syn),
                    pecahan_kategorikal=round(float(((oh > 1e-9) & (oh < 1 - 1e-9)).mean()), 4),
                    rata2_usia_sintetis_z=round(float(syn[:, 0].mean()), 3), sd_usia_sintetis_z=round(float(syn[:, 0].std()), 3)))
cek = pd.DataFrame(cek); cek.to_csv(f"{OUT}/cek_sampel_sintetis.csv", index=False); cek

,Algoritma,Implementasi,n_latih,n_positif,n_sintetis,pecahan_kategorikal,rata2_usia_sintetis_z,sd_usia_sintetis_z
0,SMOTE,implementasi sendiri,7776,3888,3689,0.1111,1.109,0.445
1,SMOTE,imbalanced-learn,7776,3888,3689,0.1094,1.105,0.457
2,SMOTE-NC,implementasi sendiri,7776,3888,3689,0.0000,1.095,0.466
3,SMOTE-NC,imbalanced-learn,7776,3888,3689,0.0000,1.099,0.474


## 4. Perbandingan kinerja: 5-fold CV × 5 seed

In [5]:
t0 = time.time(); rows = []
for seed in range(42, 47):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    for mname, est in MODELS.items():
        for (alg, impl), f in VARIAN.items():
            oof = np.zeros(len(y))
            for tr, te in cv.split(X, y):
                oof[te] = f(clone(est), seed).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
            a, b = calib(oof, y)
            rows.append(dict(seed=seed, Model=mname, Algoritma=alg, Implementasi=impl, ROC_AUC=roc_auc_score(y, oof),
                             PR_AUC=average_precision_score(y, oof), Brier=brier_score_loss(y, oof), Intercept=a, Slope=b,
                             Recall_05=recall_score(y, (oof >= .5).astype(int)), MCC_05=matthews_corrcoef(y, (oof >= .5).astype(int))))
        print(f"seed {seed} {mname} selesai  ({time.time() - t0:.0f} s)")
H = pd.DataFrame(rows); H.to_csv(f"{OUT}/hasil_per_seed.csv", index=False)

seed 42 LR selesai  (21 s)
seed 42 RF selesai  (61 s)
seed 42 HGB selesai  (95 s)
seed 43 LR selesai  (116 s)
seed 43 RF selesai  (160 s)
seed 43 HGB selesai  (194 s)
seed 44 LR selesai  (216 s)
seed 44 RF selesai  (261 s)
seed 44 HGB selesai  (296 s)
seed 45 LR selesai  (318 s)
seed 45 RF selesai  (361 s)
seed 45 HGB selesai  (395 s)
seed 46 LR selesai  (417 s)
seed 46 RF selesai  (460 s)
seed 46 HGB selesai  (496 s)


## 5. Ringkasan
- **Rata-rata ± SD antar-seed** untuk setiap implementasi.
- **Selisih** = imbalanced-learn − implementasi sendiri pada seed yang sama (rata-rata dan rentang).
- **Rasio** = |rata-rata selisih| dibagi SD antar-seed implementasi sendiri. Rasio < 1 berarti perbedaan antarimplementasi lebih kecil daripada variasi akibat pergantian seed.

In [6]:
MET = ["ROC_AUC", "PR_AUC", "Brier", "Intercept", "Slope", "Recall_05", "MCC_05"]
g = H.groupby(["Algoritma", "Model", "Implementasi"])[MET]
ringkas = (g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str)).reset_index()
ringkas.to_csv(f"{OUT}/ringkasan_mean_sd.csv", index=False); display(ringkas)

P = H.pivot_table(index=["Algoritma", "Model", "seed"], columns="Implementasi", values=MET)
sel = []
for (alg, m), d in P.groupby(level=[0, 1]):
    r = dict(Algoritma=alg, Model=m)
    for k in ["ROC_AUC", "PR_AUC", "Brier"]:
        diff = d[(k, "imbalanced-learn")] - d[(k, "implementasi sendiri")]
        sd = d[(k, "implementasi sendiri")].std()
        r[f"Δ{k} rata2"] = round(diff.mean(), 4); r[f"Δ{k} min..maks"] = f"{diff.min():.4f}..{diff.max():.4f}"
        r[f"rasio {k}"] = round(abs(diff.mean()) / sd, 2) if sd > 0 else np.nan
    sel.append(r)
sel = pd.DataFrame(sel); sel.to_csv(f"{OUT}/selisih_implementasi.csv", index=False); display(sel)
print("Selisih absolut terbesar: ROC-AUC", H.pipe(lambda _: sel["ΔROC_AUC rata2"].abs().max()),
      "| PR-AUC", sel["ΔPR_AUC rata2"].abs().max(), "| Brier", sel["ΔBrier rata2"].abs().max())

,Algoritma,Model,Implementasi,ROC_AUC,PR_AUC,Brier,Intercept,Slope,Recall_05,MCC_05
0,SMOTE,HGB,imbalanced-learn,0.791 ± 0.003,0.128 ± 0.003,0.06 ± 0.001,-0.878 ± 0.016,0.439 ± 0.013,0.133 ± 0.013,0.099 ± 0.013
1,SMOTE,HGB,implementasi sendiri,0.787 ± 0.006,0.124 ± 0.005,0.061 ± 0.0,-0.891 ± 0.023,0.429 ± 0.013,0.133 ± 0.016,0.096 ± 0.013
2,SMOTE,LR,imbalanced-learn,0.832 ± 0.004,0.18 ± 0.005,0.171 ± 0.001,-3.057 ± 0.009,0.774 ± 0.018,0.785 ± 0.005,0.25 ± 0.002
3,SMOTE,LR,implementasi sendiri,0.832 ± 0.003,0.179 ± 0.007,0.171 ± 0.001,-3.053 ± 0.01,0.781 ± 0.018,0.787 ± 0.009,0.251 ± 0.004
4,SMOTE,RF,imbalanced-learn,0.793 ± 0.005,0.122 ± 0.003,0.061 ± 0.001,-1.084 ± 0.015,0.409 ± 0.013,0.087 ± 0.01,0.059 ± 0.015
5,SMOTE,RF,implementasi sendiri,0.795 ± 0.003,0.123 ± 0.003,0.061 ± 0.001,-1.083 ± 0.024,0.404 ± 0.012,0.086 ± 0.01,0.058 ± 0.009
6,SMOTE-NC,HGB,imbalanced-learn,0.782 ± 0.003,0.124 ± 0.004,0.082 ± 0.001,-1.658 ± 0.033,0.404 ± 0.003,0.291 ± 0.025,0.145 ± 0.013
7,SMOTE-NC,HGB,implementasi sendiri,0.783 ± 0.003,0.125 ± 0.003,0.081 ± 0.001,-1.616 ± 0.028,0.409 ± 0.009,0.288 ± 0.017,0.147 ± 0.009
8,SMOTE-NC,LR,imbalanced-learn,0.798 ± 0.007,0.153 ± 0.004,0.167 ± 0.001,-2.981 ± 0.016,0.601 ± 0.029,0.675 ± 0.019,0.2 ± 0.01
9,SMOTE-NC,LR,implementasi sendiri,0.8 ± 0.007,0.153 ± 0.004,0.17 ± 0.002,-2.989 ± 0.012,0.625 ± 0.033,0.707 ± 0.019,0.211 ± 0.008


,Algoritma,Model,ΔROC_AUC rata2,ΔROC_AUC min..maks,rasio ROC_AUC,ΔPR_AUC rata2,ΔPR_AUC min..maks,rasio PR_AUC,ΔBrier rata2,ΔBrier min..maks,rasio Brier
0,SMOTE,HGB,0.0047,-0.0055..0.0105,0.75,0.0034,-0.0032..0.0115,0.71,-0.0006,-0.0016..0.0002,1.67
1,SMOTE,LR,-0.0001,-0.0015..0.0013,0.04,0.0002,-0.0029..0.0042,0.02,-0.0004,-0.0012..0.0010,0.28
2,SMOTE,RF,-0.0012,-0.0053..0.0023,0.34,-0.0010,-0.0039..0.0011,0.37,0.0002,-0.0002..0.0004,0.19
3,SMOTE-NC,HGB,-0.0016,-0.0071..0.0034,0.51,-0.0008,-0.0046..0.0034,0.29,0.0016,-0.0006..0.0038,1.27
4,SMOTE-NC,LR,-0.0013,-0.0045..0.0009,0.18,0.0002,-0.0032..0.0039,0.06,-0.0034,-0.0047..-0.0019,2.08
5,SMOTE-NC,RF,-0.0009,-0.0052..0.0031,0.22,0.0012,-0.0011..0.0036,0.54,0.0025,0.0020..0.0032,2.65


Selisih absolut terbesar: ROC-AUC 0.0047 | PR-AUC 0.0034 | Brier 0.0034


## 6. Catatan untuk interpretasi
1. Kedua SMOTE seharusnya menghasilkan sampel sintetis dengan sifat yang sama (Bagian 3: jumlah sampel sama, proporsi nilai pecahan pada kolom one-hot hampir sama). Urutan bilangan acak berbeda, sehingga sampelnya tidak identik.
2. SMOTE-NC versi naskah bekerja pada ruang one-hot dengan penalti median SD per atribut kategorikal, sedangkan imbalanced-learn bekerja pada kode kategori. Selisih kecil dapat berasal dari perbedaan definisi jarak ini.
3. Implementasi dianggap setara apabila selisih rata-rata ROC-AUC dan PR-AUC kecil (misalnya < 0,01) dan rasio terhadap SD antar-seed sekitar 1 atau kurang. Kirimkan file di folder `hasil_verifikasi_imblearn/` untuk dituliskan ke naskah.